# TinyCeNN Qwen3.5: test → GGUF → Hugging Face

This Colab tests these three models, attempts a faithful GGUF conversion, validates the result with `llama-cli`, and uploads successful conversions as new Hugging Face repos:

- `vtava/Qwen3.5-0.8B-MemoryFusion` → `vtava/Qwen3.5-0.8B-MemoryFusion-GGUF`
- `vtava/Qwen3.5-0.8B-CeNN-Integrated-V1` → `vtava/Qwen3.5-0.8B-CeNN-Integrated-V1-GGUF`
- `vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32` → `vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32-GGUF`

> **Strict validation:** custom TinyCeNN tensors are never discarded and base Qwen weights are never substituted. A model is uploaded only if the current `llama.cpp` converter accepts it and `llama-cli` can execute the generated `Q4_K_M` GGUF. Adapter/custom architectures that are not yet implemented in llama.cpp are reported clearly instead of producing a misleading GGUF.


In [ ]:
#@title 1. Install Python dependencies
%pip -q install -U huggingface_hub accelerate safetensors sentencepiece protobuf
%pip -q install -U "transformers @ git+https://github.com/huggingface/transformers.git@main"


In [ ]:
#@title 2. Clone TinyCeNN-LM + latest llama.cpp and build llama.cpp
from pathlib import Path
import subprocess, sys

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)

root = Path('/content')
tiny = root/'TinyCeNN-LM'
llama = root/'llama.cpp'
if tiny.exists(): run(['git','pull','--ff-only'], tiny)
else: run(['git','clone','--depth','1','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(tiny)])
if llama.exists(): run(['git','pull','--ff-only'], llama)
else: run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',str(llama)])
req = llama/'requirements.txt'
if req.exists(): run([sys.executable,'-m','pip','install','-q','-r',str(req)])
run(['cmake','-S',str(llama),'-B',str(llama/'build'),'-DGGML_CUDA=OFF','-DLLAMA_CURL=OFF','-DCMAKE_BUILD_TYPE=Release'])
run(['cmake','--build',str(llama/'build'),'--config','Release','-j','2'])
print('TinyCeNN commit:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=tiny,text=True).strip())
print('llama.cpp commit:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=llama,text=True).strip())


In [ ]:
#@title 3. Hugging Face login
# Recommended: add a Colab Secret named HF_TOKEN with WRITE permission for vtava.
import os
from getpass import getpass
from huggingface_hub import HfApi, login
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not token: token = getpass('HF write token: ').strip()
login(token=token, add_to_git_credential=False)
os.environ['HF_TOKEN'] = token
print('Authenticated as:', HfApi(token=token).whoami()['name'])


In [ ]:
#@title 4. Test all 3 models, convert to F16 + Q4_K_M GGUF, validate, upload
# Set --no-keep-f16 if you only want the smaller Q4_K_M artifact uploaded.
cmd = [
    sys.executable, str(tiny/'scripts/qwen35_gguf_export.py'),
    '--llama-dir', str(llama),
    '--work-dir', '/content/tinycenn_gguf_export',
    '--token', token,
    '--keep-f16',
    '--upload',
]
subprocess.run(cmd, check=True)


In [ ]:
#@title 5. Show final machine-readable report
import json
from IPython.display import display
import pandas as pd
report_path = Path('/content/tinycenn_gguf_export/all_results.json')
results = json.loads(report_path.read_text())
display(pd.DataFrame([{
    'source': r.get('source'),
    'source_test': r.get('source_test_ok'),
    'conversion': r.get('conversion_ok'),
    'llama_test': r.get('gguf_test_ok'),
    'uploaded': r.get('uploaded'),
    'target': r.get('url') or r.get('target'),
    'error': r.get('conversion_error') or r.get('source_test_error') or r.get('upload_error') or r.get('fatal_error'),
} for r in results]))
print('Full report:', report_path)
